My aim is to understand how PyTorch Pipeline is written. I'm not focusing on optimized code right now. This part will teach me framework.

I'll do following steps:
- Load the Dataset
- Basic Preprocessing
- Training Process
    - Create the Model
    - Forward Pass
    - Loss Calculation
    - Backpropagation
    - Parameter Update
- Model Evaluation

In [1]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [2]:
df = pd.read_csv("https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv")
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [3]:
print(df.shape)

(569, 33)


In [4]:
# Dropping No-Use Columns
df.drop(["id", "Unnamed: 32"], axis=1, inplace=True)

In [5]:
df.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [6]:
# Train Test Split
X_train, X_test, y_train, y_test = train_test_split(df.drop(["diagnosis"], axis=1), df["diagnosis"], test_size=0.2, random_state=42)

In [7]:
# Scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [8]:
X_train

array([[-1.44075296, -0.43531947, -1.36208497, ...,  0.9320124 ,
         2.09724217,  1.88645014],
       [ 1.97409619,  1.73302577,  2.09167167, ...,  2.6989469 ,
         1.89116053,  2.49783848],
       [-1.39998202, -1.24962228, -1.34520926, ..., -0.97023893,
         0.59760192,  0.0578942 ],
       ...,
       [ 0.04880192, -0.55500086, -0.06512547, ..., -1.23903365,
        -0.70863864, -1.27145475],
       [-0.03896885,  0.10207345, -0.03137406, ...,  1.05001236,
         0.43432185,  1.21336207],
       [-0.54860557,  0.31327591, -0.60350155, ..., -0.61102866,
        -0.3345212 , -0.84628745]], shape=(455, 30))

In [9]:
y_train

68     B
181    M
63     B
248    B
60     B
      ..
71     B
106    B
270    B
435    M
102    B
Name: diagnosis, Length: 455, dtype: str

In [10]:
# Label Encoding
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [11]:
y_train

array([0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1,
       0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0,
       0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0,
       0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0,
       1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1,
       0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0,
       1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0,
       0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,
       0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1,
       0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0,
       1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0,
       0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1,
       0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1,
       1, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0,

In [12]:
# Move from Numpy Arrays to PyTorch Tensors
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

X_train_tensor = torch.from_numpy(X_train).to(device=device)
X_test_tensor = torch.from_numpy(X_test).to(device=device)
y_train_tensor = torch.from_numpy(y_train).to(device=device)
y_test_tensor = torch.from_numpy(y_test).to(device=device)


In [13]:
X_train_tensor.shape

torch.Size([455, 30])

In [14]:
y_train_tensor.shape

torch.Size([455])

In [15]:
X_train_tensor

tensor([[-1.4408, -0.4353, -1.3621,  ...,  0.9320,  2.0972,  1.8865],
        [ 1.9741,  1.7330,  2.0917,  ...,  2.6989,  1.8912,  2.4978],
        [-1.4000, -1.2496, -1.3452,  ..., -0.9702,  0.5976,  0.0579],
        ...,
        [ 0.0488, -0.5550, -0.0651,  ..., -1.2390, -0.7086, -1.2715],
        [-0.0390,  0.1021, -0.0314,  ...,  1.0500,  0.4343,  1.2134],
        [-0.5486,  0.3133, -0.6035,  ..., -0.6110, -0.3345, -0.8463]],
       device='cuda:0', dtype=torch.float64)

In [16]:
y_train_tensor

tensor([0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0,
        0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0,
        0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1,
        0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1,
        0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0,
        1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1,
        0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0,
        0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1,
        1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1,
        0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0,
        0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1,
        0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1,
        0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0,

In [ ]:
# Defining the Model

# Most Basic Neural Network
class SimpleNN():
    if torch.cuda.is_available():
        device = torch.device("cpu")
    else:
        device = torch.device("cpu")
    def __init__(self, X):
        self.weights = torch.rand(X.shape[1], 1, dtype=torch.float64, requires_grad = True, device=device)
        self.bias = torch.zeros(1, dtype=torch.float64, requires_grad = True, device=device)

    def forward(self, X):
        z = X@self.weights + self.bias
        y_pred = torch.sigmoid(z)
        return y_pred

    def loss_function(self, y_pred, y_train):
        # Clamp predictions to avoid log(0)
        epsilon = 1e-7
        y_pred = torch.clamp(y_pred, epsilon, 1-epsilon)
        
        loss = -(y_train * torch.log(y_pred) + (1 - y_train) * torch.log(1 - y_pred)).mean()
        return loss 

        

In [18]:
# Training Pipeline

learning_rate = 0.1
epochs = 40
model = SimpleNN(X_train_tensor)

# Training
# Define Loop for Epochs
for epoch in range(epochs):
    # Forward Pass
    y_pred = model.forward(X_train_tensor)

    # Compute Loss
    loss = model.loss_function(y_pred, y_train_tensor)

    # Backpropagation / Backward Pass
    loss.backward()

    # Update Parameters
    with torch.no_grad():
        model.weights -= learning_rate * model.weights.grad
        model.bias -= learning_rate * model.bias.grad

    # Zero Grad
    model.weights.grad.zero_()
    model.bias.grad.zero_()
    print(f"Epoch: {epoch+1}, Loss: {loss}")

    

Epoch: 1, Loss: 3.3586477550501983
Epoch: 2, Loss: 3.2182608197026332
Epoch: 3, Loss: 3.074500108691241
Epoch: 4, Loss: 2.9298916073784924
Epoch: 5, Loss: 2.782911005572013
Epoch: 6, Loss: 2.6222837505472025
Epoch: 7, Loss: 2.446851214947929
Epoch: 8, Loss: 2.2685973093147065
Epoch: 9, Loss: 2.091608026534823
Epoch: 10, Loss: 1.918368475481747
Epoch: 11, Loss: 1.7456646128733595
Epoch: 12, Loss: 1.5761811970877777
Epoch: 13, Loss: 1.4168793056090079
Epoch: 14, Loss: 1.2680665495517065
Epoch: 15, Loss: 1.1370190025458586
Epoch: 16, Loss: 1.02626565268484
Epoch: 17, Loss: 0.9370786079226856
Epoch: 18, Loss: 0.8689540883501657
Epoch: 19, Loss: 0.8196367081537768
Epoch: 20, Loss: 0.7856625115125995
Epoch: 21, Loss: 0.7631191139193446
Epoch: 22, Loss: 0.7483715265073134
Epoch: 23, Loss: 0.7385547312692642
Epoch: 24, Loss: 0.7317017392799297
Epoch: 25, Loss: 0.726592556236939
Epoch: 26, Loss: 0.7225181562188985
Epoch: 27, Loss: 0.7190810991020268
Epoch: 28, Loss: 0.7160618258997805
Epoch: 29

In [19]:
model.weights

tensor([[ 0.0153],
        [ 0.1552],
        [-0.0925],
        [-0.0991],
        [ 0.2884],
        [-0.2438],
        [ 0.0156],
        [-0.2096],
        [-0.0152],
        [ 0.0363],
        [-0.1491],
        [ 0.1095],
        [ 0.0860],
        [ 0.1482],
        [-0.0187],
        [-0.2877],
        [-0.1595],
        [ 0.2192],
        [ 0.0142],
        [ 0.2493],
        [ 0.3796],
        [-0.1518],
        [ 0.0408],
        [ 0.0054],
        [-0.2534],
        [-0.4165],
        [-0.0622],
        [ 0.1858],
        [ 0.3023],
        [ 0.3410]], device='cuda:0', dtype=torch.float64, requires_grad=True)

In [20]:
model.bias

tensor([-0.2670], device='cuda:0', dtype=torch.float64, requires_grad=True)

In [21]:
# Evaluation
with torch.no_grad():
    y_pred = model.forward(X_test_tensor)
    y_pred = (y_pred > 0.9).float()
    accuracy = (y_pred == y_test_tensor).float().mean()
    print(f"Accuracy: {accuracy.item()}")

Accuracy: 0.6228070259094238
